# LuCY Kernel — ARC-AGI-3 Submission

**LuCY** (Language Understanding Cognitive sYstem) — cognitive kernel evaluation on ARC-AGI-3 puzzles.

- CERTIFIED_100=false | unique_human_proven=false | DEBUG=true
- Architecture: 8 cognitive operators (SE/SI/NE/NI/TE/TI/FE/FI) + Arbitrator
- Strategy: Each ARC puzzle is encoded as a structured text stimulus
- Output: Predicted output grids based on cognitive pattern analysis
- Competition: **arc-prize-2026-arc-agi-3**

**Note scientifique honnête:** LuCY Phase 4 est un moteur de raisonnement cognitif symbolique.
Les puzzles ARC-AGI-3 nécessitent de la reconnaissance de motifs visuels/abstraits que les 8 opérateurs
actuels ne modélisent pas encore directement. Ce notebook documente le comportement réel du kernel.


In [ ]:
# ============================================================
# CELL 1 — Environment setup & imports
# CERTIFIED_100=false | unique_human_proven=false | DEBUG=true
# ============================================================
import os
import sys
import json
import math
import time
import hashlib
import itertools
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Optional, List, Dict, Tuple
from collections import Counter, defaultdict

EVAL_TS = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
CERTIFIED_100 = False
UNIQUE_HUMAN_PROVEN = False
DEBUG = True

print(f"LuCY ARC-AGI-3 Submission — {EVAL_TS}")
COMPETITION = "arc-prize-2026-arc-agi-3"
print(f"CERTIFIED_100={CERTIFIED_100} | unique_human_proven={UNIQUE_HUMAN_PROVEN} | DEBUG={DEBUG}")
print(f"Python: {sys.version}")

In [ ]:
# ============================================================
# CELL 2 — Locate ARC-AGI-3 dataset files
# ============================================================
import glob

# Kaggle competition datasets are mounted at /kaggle/input/
INPUT_DIR = Path('/kaggle/input')
COMPETITION_ID = "arc-prize-2026-arc-agi-3"

# Search for ARC-AGI-3 specific test files first
test_files = (
    list(INPUT_DIR.rglob('arc_agi_3_test_challenges.json')) +
    list(INPUT_DIR.rglob('arc-agi-3-test-challenges.json')) +
    list(INPUT_DIR.rglob('*arc*3*test*.json')) +
    list(INPUT_DIR.rglob('*test*challenges*.json'))
)

# Also look for sample submission
sample_files = (
    list(INPUT_DIR.rglob('sample_submission*.csv')) +
    list(INPUT_DIR.rglob('sample_submission*.json'))
)

print("=== Input directory contents ===")
for p in sorted(INPUT_DIR.rglob('*')):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {p.relative_to(INPUT_DIR)} ({size:,} bytes)")

print(f"\nTest files found: {len(test_files)}")
for f in test_files:
    print(f"  {f}")

print(f"\nSample submission files found: {len(sample_files)}")
for f in sample_files:
    print(f"  {f}")


In [ ]:
# ============================================================
# CELL 3 — Load test puzzles
# ============================================================

def load_test_puzzles(input_dir: Path) -> Dict[str, Any]:
    """Load test puzzles from the competition dataset."""
    # Try multiple known paths for ARC competitions
    candidate_paths = [
        # ARC-AGI-3 paths (priority)
        input_dir / 'arc-prize-2026-arc-agi-3' / 'arc_agi_3_test_challenges.json',
        input_dir / 'arc-prize-2026-arc-agi-3' / 'arc-agi-3-test-challenges.json',
        input_dir / 'arc-prize-2026-arc-agi-3' / 'test_challenges.json',
        input_dir / 'arc-prize-2026-arc-agi-3' / 'test.json',
        # Generic fallback
        input_dir / 'arc-prize-2026-arc-agi-2' / 'arc_agi_2_test_challenges.json',
    ]
    # Generic search
    for p in input_dir.rglob('*.json'):
        candidate_paths.append(p)

    for path in candidate_paths:
        if path.exists():
            try:
                with open(path) as f:
                    data = json.load(f)
                if isinstance(data, dict) and len(data) > 0:
                    # Check if it's a test challenges file (no 'output' in test pairs)
                    sample_key = next(iter(data))
                    sample_val = data[sample_key]
                    if isinstance(sample_val, dict) and 'test' in sample_val:
                        print(f"[OK] Loaded test puzzles from: {path}")
                        print(f"     Puzzles count: {len(data)}")
                        return data
            except Exception as e:
                print(f"[SKIP] {path}: {e}")

    print("[WARNING] No test puzzles found — will use empty dict")
    return {}


def load_sample_submission(input_dir: Path) -> Dict[str, Any]:
    """Load sample submission to get expected format."""
    for p in input_dir.rglob('sample_submission.json'):
        with open(p) as f:
            data = json.load(f)
        print(f"[OK] Sample submission loaded from: {p}")
        print(f"     Keys: {list(data.keys())[:3]}...")
        return data
    return {}


test_challenges = load_test_puzzles(INPUT_DIR)
sample_submission = load_sample_submission(INPUT_DIR)

print(f"\nTest puzzles loaded: {len(test_challenges)}")
if test_challenges:
    first_id = next(iter(test_challenges))
    print(f"First puzzle ID: {first_id}")
    print(f"Keys: {list(test_challenges[first_id].keys())}")

In [ ]:
# ============================================================
# CELL 4 — LuCY Cognitive Kernel (self-contained, no external deps)
# Architecture: 8 operators + Arbitrator + Pattern Extractor (Phase 5 bridge)
# CERTIFIED_100=false | unique_human_proven=false
# ============================================================

class LuCYArcSolver:
    """
    LuCY cognitive kernel adapted for ARC-AGI grid prediction.

    Phase 4 architecture (8 operators) + PE bridge layer for grid output.
    Honest note: Phase 4 is a symbolic cognitive engine.
    Grid prediction uses rule-based pattern extraction as a bridge
    until native PE (Pattern Extraction) operator is implemented.
    """

    def __init__(self):
        self.certified_100 = False
        self.unique_human_proven = False
        self.debug = True
        self._cycles = 0
        self._conflict_count = 0
        if self.debug:
            print("[LuCY] Kernel initialized | CERTIFIED_100=false | DEBUG=true")

    # ----------------------------------------------------------------
    # Pattern Extraction layer (PE — bridge toward Phase 5)
    # ----------------------------------------------------------------

    def _extract_grid_patterns(self, train_pairs: List[Dict]) -> Dict[str, Any]:
        """Extract transformation patterns from training examples."""
        patterns = {
            'color_mapping': {},      # input_color -> output_color
            'size_change': None,      # 'same', 'smaller', 'larger'
            'color_counts': [],       # (input_count, output_count) per pair
            'unique_colors_in': set(),
            'unique_colors_out': set(),
            'output_sizes': [],       # (rows, cols) of outputs
            'input_sizes': [],
        }

        color_map_votes = defaultdict(Counter)  # color_map_votes[in_color][out_color]

        for pair in train_pairs:
            inp = pair.get('input', [])
            out = pair.get('output', [])
            if not inp or not out:
                continue

            inp_flat = [c for row in inp for c in row]
            out_flat = [c for row in out for c in row]

            patterns['input_sizes'].append((len(inp), len(inp[0]) if inp else 0))
            patterns['output_sizes'].append((len(out), len(out[0]) if out else 0))

            in_colors = set(inp_flat)
            out_colors = set(out_flat)
            patterns['unique_colors_in'].update(in_colors)
            patterns['unique_colors_out'].update(out_colors)

            # Color frequency mapping
            in_counter = Counter(inp_flat)
            out_counter = Counter(out_flat)

            # Try to find direct color mapping by position
            if len(inp) == len(out) and all(len(r1) == len(r2) for r1, r2 in zip(inp, out)):
                for r_in, r_out in zip(inp, out):
                    for c_in, c_out in zip(r_in, r_out):
                        color_map_votes[c_in][c_out] += 1

        # Build majority-vote color mapping
        for in_c, votes in color_map_votes.items():
            if votes:
                patterns['color_mapping'][in_c] = votes.most_common(1)[0][0]

        # Determine size change pattern
        if patterns['input_sizes'] and patterns['output_sizes']:
            in_areas = [r * c for r, c in patterns['input_sizes']]
            out_areas = [r * c for r, c in patterns['output_sizes']]
            avg_ratio = sum(o / i for i, o in zip(in_areas, out_areas) if i > 0) / max(len(in_areas), 1)
            if abs(avg_ratio - 1.0) < 0.1:
                patterns['size_change'] = 'same'
            elif avg_ratio > 1.1:
                patterns['size_change'] = 'larger'
            else:
                patterns['size_change'] = 'smaller'

            # Most common output size
            if patterns['output_sizes']:
                size_counter = Counter(patterns['output_sizes'])
                patterns['most_common_output_size'] = size_counter.most_common(1)[0][0]

        return patterns

    def _apply_transformation(self, test_input: List[List[int]], patterns: Dict) -> List[List[int]]:
        """Apply extracted patterns to predict output grid."""
        # Determine output size
        in_rows = len(test_input)
        in_cols = len(test_input[0]) if test_input else 0

        if 'most_common_output_size' in patterns:
            out_rows, out_cols = patterns['most_common_output_size']
        elif patterns['size_change'] == 'same':
            out_rows, out_cols = in_rows, in_cols
        else:
            # Fallback: same size
            out_rows, out_cols = in_rows, in_cols

        color_map = patterns.get('color_mapping', {})

        # Apply color mapping if available and sizes match
        if color_map and out_rows == in_rows and out_cols == in_cols:
            output = []
            for row in test_input:
                new_row = [color_map.get(c, c) for c in row]
                output.append(new_row)
            return output

        # Fallback: return copy of input with size adjustment
        output = []
        for r in range(out_rows):
            row = []
            for c in range(out_cols):
                if r < in_rows and c < in_cols:
                    row.append(test_input[r][c])
                else:
                    row.append(0)  # pad with black
            output.append(row)
        return output

    # ----------------------------------------------------------------
    # 8 Cognitive operators (simplified, self-contained)
    # ----------------------------------------------------------------

    def _se(self, stimulus: str) -> Dict:
        """Extraverted Sensing — extract concrete features."""
        length = len(stimulus)
        digit_count = sum(1 for c in stimulus if c.isdigit())
        return {'length': length, 'digit_ratio': digit_count / max(length, 1)}

    def _si(self, stimulus: str, se_result: Dict) -> Dict:
        """Introverted Sensing — compare to stored patterns."""
        familiarity = 0.5 if se_result['digit_ratio'] > 0.1 else 0.3
        return {'familiarity': familiarity, 'pattern_type': 'numerical_grid'}

    def _ne(self, stimulus: str) -> Dict:
        """Extraverted iNtuition — generate hypotheses."""
        hypotheses = [
            'color_permutation', 'spatial_rotation', 'size_scaling',
            'pattern_completion', 'symmetry_application'
        ]
        return {'hypotheses': hypotheses[:3], 'count': 3}  # limit to 5 max

    def _ni(self, hypotheses: List[str]) -> Dict:
        """Introverted iNtuition — converge on most likely pattern."""
        # For ARC: color transformations are most common
        preferred = 'color_permutation' if 'color_permutation' in hypotheses else hypotheses[0] if hypotheses else 'unknown'
        return {'preferred_hypothesis': preferred, 'confidence': 0.55}

    def _te(self, ni_result: Dict, se_result: Dict) -> Dict:
        """Extraverted Thinking — logical evaluation."""
        score = ni_result['confidence'] * (0.5 + se_result['digit_ratio'])
        return {'logical_score': min(1.0, score), 'decision': 'proceed'}

    def _ti(self, te_result: Dict) -> Dict:
        """Introverted Thinking — internal consistency check."""
        consistent = te_result['logical_score'] > 0.3
        return {'consistent': consistent, 'ti_score': te_result['logical_score'] * 0.9}

    def _fe(self, stimulus: str) -> Dict:
        """Extraverted Feeling — external value alignment."""
        # ARC puzzles have no explicit value/emotion context
        # This causes FE/FI conflict — documented limit of Phase 4
        return {'value_score': 0.21, 'external_alignment': 'low'}

    def _fi(self) -> Dict:
        """Introverted Feeling — internal value consistency."""
        return {'internal_value': 0.645, 'authenticity': 'neutral'}

    def _arbitrate(self, te: Dict, ti: Dict, fe: Dict, fi: Dict) -> Dict:
        """Arbitrate between cognitive functions."""
        fe_score = fe['value_score']
        fi_score = fi['internal_value']
        conflict = abs(fe_score - fi_score) > 0.3

        if conflict:
            self._conflict_count += 1

        # Dominant = TE for ARC puzzles (logical analysis)
        consensus = (te['logical_score'] + ti['ti_score']) / 2
        confidence = te['logical_score']
        composite = confidence * consensus - (0.3 if conflict else 0.0)
        composite = max(0.0, min(1.0, composite))

        return {
            'conflict_detected': conflict,
            'dominant': 'TE',
            'consensus': consensus,
            'confidence': confidence,
            'composite': composite,
        }

    # ----------------------------------------------------------------
    # Main solve method
    # ----------------------------------------------------------------

    def solve_puzzle(self, puzzle_id: str, puzzle: Dict) -> Dict[str, Any]:
        """
        Solve one ARC puzzle.
        Returns {puzzle_id: [[attempt1], [attempt2]]} format.
        """
        self._cycles += 1
        t0 = time.time()

        train = puzzle.get('train', [])
        test = puzzle.get('test', [])

        # Phase 1: Extract patterns from training examples
        patterns = self._extract_grid_patterns(train)

        # Phase 2: Run 8-operator cognitive cycle on stimulus
        stimulus = json.dumps({'id': puzzle_id, 'patterns': str(patterns)[:200]})
        se_r = self._se(stimulus)
        si_r = self._si(stimulus, se_r)
        ne_r = self._ne(stimulus)
        ni_r = self._ni(ne_r['hypotheses'])
        te_r = self._te(ni_r, se_r)
        ti_r = self._ti(te_r)
        fe_r = self._fe(stimulus)
        fi_r = self._fi()
        arb = self._arbitrate(te_r, ti_r, fe_r, fi_r)

        # Phase 3: Generate predictions for each test input
        attempts_per_test = []
        for test_pair in test:
            test_input = test_pair.get('input', [[0]])
            # Attempt 1: apply extracted patterns
            attempt1 = self._apply_transformation(test_input, patterns)
            # Attempt 2: identity (copy input as fallback)
            attempt2 = [row[:] for row in test_input]
            attempts_per_test.append([attempt1, attempt2])

        dur_ms = (time.time() - t0) * 1000

        return {
            'puzzle_id': puzzle_id,
            'attempts': attempts_per_test,
            'composite_score': arb['composite'],
            'confidence': arb['confidence'],
            'conflict': arb['conflict_detected'],
            'duration_ms': dur_ms,
            'patterns_found': len(patterns.get('color_mapping', {})),
        }


# Test the solver
solver = LuCYArcSolver()
print(f"Solver ready | cycles=0 | CERTIFIED_100={solver.certified_100}")

In [ ]:
# ============================================================
# CELL 5 — Solve all test puzzles
# ============================================================

def solve_all_puzzles(solver: LuCYArcSolver, challenges: Dict) -> Dict:
    """
    Solve all test puzzles and build the submission dictionary.
    Submission format: {puzzle_id: [{"attempt_1": grid, "attempt_2": grid}]}
    """
    submission = {}
    stats = {'total': 0, 'with_patterns': 0, 'identity_fallback': 0, 'conflict': 0}
    n = len(challenges)

    for i, (puzzle_id, puzzle) in enumerate(challenges.items()):
        if (i + 1) % 50 == 0 or i == 0:
            pct = (i + 1) / max(n, 1) * 100
            print(f"  [{pct:.0f}%] {i+1}/{n} — {puzzle_id}")

        result = solver.solve_puzzle(puzzle_id, puzzle)
        stats['total'] += 1
        if result['patterns_found'] > 0:
            stats['with_patterns'] += 1
        if result['conflict']:
            stats['conflict'] += 1

        # Build submission for this puzzle
        test_outputs = []
        for attempts in result['attempts']:
            attempt1 = attempts[0] if len(attempts) > 0 else [[0]]
            attempt2 = attempts[1] if len(attempts) > 1 else [[0]]
            test_outputs.append({
                'attempt_1': attempt1,
                'attempt_2': attempt2,
            })

        submission[puzzle_id] = test_outputs

    print(f"\n=== Solve stats ===")
    print(f"  Total puzzles: {stats['total']}")
    print(f"  With color patterns: {stats['with_patterns']} ({stats['with_patterns']/max(stats['total'],1)*100:.1f}%)")
    print(f"  FE/FI conflict: {stats['conflict']} ({stats['conflict']/max(stats['total'],1)*100:.1f}%)")

    return submission


print("Solving all test puzzles...")
if test_challenges:
    submission = solve_all_puzzles(solver, test_challenges)
    print(f"\nSubmission built: {len(submission)} puzzles")
    # Show a sample
    if submission:
        sample_id = next(iter(submission))
        sample_out = submission[sample_id]
        print(f"Sample [{sample_id}]: {len(sample_out)} test cases")
        if sample_out:
            a1 = sample_out[0]['attempt_1']
            print(f"  attempt_1 shape: {len(a1)} x {len(a1[0]) if a1 else 0}")
else:
    print("[WARNING] No test challenges loaded — creating empty submission")
    submission = {}

In [ ]:
# ============================================================
# CELL 6 — Write submission.json
# ============================================================

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission_path = OUTPUT_DIR / 'submission.json'

with open(submission_path, 'w') as f:
    json.dump(submission, f)

file_size = submission_path.stat().st_size
print(f"[OK] submission.json written: {file_size:,} bytes")
print(f"     Path: {submission_path}")
print(f"     Puzzles: {len(submission)}")

# Validate format
print("\n=== Format validation ===")
errors = 0
for pid, test_outs in submission.items():
    if not isinstance(test_outs, list):
        print(f"  ERROR [{pid}]: expected list, got {type(test_outs)}")
        errors += 1
        continue
    for j, t in enumerate(test_outs):
        if not isinstance(t, dict):
            print(f"  ERROR [{pid}][{j}]: expected dict, got {type(t)}")
            errors += 1
        elif 'attempt_1' not in t or 'attempt_2' not in t:
            print(f"  ERROR [{pid}][{j}]: missing attempt_1 or attempt_2")
            errors += 1

if errors == 0:
    print(f"  Format OK — 0 errors on {len(submission)} puzzles")
else:
    print(f"  {errors} format errors detected")

print(f"\nCERTIFIED_100={CERTIFIED_100} | unique_human_proven={UNIQUE_HUMAN_PROVEN}")

In [ ]:
# ============================================================
# CELL 7 — Forensic summary & diagnostics
# CERTIFIED_100=false — résultats honnêtes
# ============================================================

print("=" * 60)
print("LuCY ARC-AGI-3 Evaluation Summary")
print("=" * 60)
print(f"Timestamp: {EVAL_TS}")
print(f"CERTIFIED_100: {CERTIFIED_100}")
print(f"unique_human_proven: {UNIQUE_HUMAN_PROVEN}")
print()
print("=== Architecture ===")
print("Phase 4 kernel: 8 operators (SE/SI/NE/NI/TE/TI/FE/FI) + Arbitrator")
print("Bridge layer: PE (Pattern Extraction) — rule-based color/size mapping")
print("Limitation: FE/FI conflict systematic on abstract numeric grids")
print("           (no emotional/value context in ARC puzzles)")
print()
print("=== Known limits of Phase 4 on ARC-AGI ===")
print("1. Systematic FE/FI conflict → cognitive composite score ~0.125")
print("2. No native pattern induction (PE operator not yet implemented)")
print("3. Color mapping by position: works for simple transformations only")
print("4. No spatial reasoning (rotation, reflection, translation)")
print("5. No object detection or segmentation")
print()
print("=== Phase 5 roadmap (not yet implemented) ===")
print("- PE operator: Pattern Extraction (visual/grid native)")
print("- PI operator: Pattern Induction (generalize from examples)")
print("- Spatial memory integration")
print("- FE/FI adaptation for abstract non-verbal stimuli")
print()
print(f"Total cognitive cycles: {solver._cycles}")
print(f"FE/FI conflicts: {solver._conflict_count} / {solver._cycles}")
print(f"Conflict rate: {solver._conflict_count/max(solver._cycles,1)*100:.1f}%")
print()
print(f"submission.json: {len(submission)} puzzles — ready for ARC-AGI-3 evaluation")
print(f"Competition: arc-prize-2026-arc-agi-3")